# P2C-64 v4.0 (`SWAR_V40`: Scenario 3 — 100%-Leaf On-Demand SWAR-Packed + Fixed-Depth Relabel & Unfrozen Tier-1 + Fair Search Compression)

`docs/P2C64_PLATEAU_BREAKTHROUGH_REPORT_JA.md`（Victory Audit 通過済み）に基づき、純スカラー（0-SIMD）・完全公平対局で **`+45 〜 +63 Elo` 帯突破** を狙う **P2C-64 v4.0** を **Gate 0 → Gate 1 → Gate 2** の明確な Pass/Kill 判定付きで実行するワンクリック Google Colab ノートブックです。

### v4.0 の 3 軸統合アーキテクチャ
1. **軸 A（Path A-I / `SWAR_V40`）**：ゲート・Stage 2.5・Stage 3・Flood-Fill・着手差分状態を全撤去し、オフセット符号化 `int16x4` SWAR（1-sub PTM, bit-63 安全 ReLU）で **100% 全葉単一パス評価**（42 カーネル `0 xmm/ymm/zmm`, `0 PLT`, `0 div`）。
2. **軸 B（Gate 0 + Gate 1）**：既存 `train.bin` / `val.bin` を無改変のまま `ply 12..49` を固定深さ `d6` サイドカー `.i16` へ OpenMP 再ラベルし、凍結解除 Tier-1 ＋ `SWAR_V40` を End-to-End 共同学習（対照 `raw_ntuple64_relabel.p2c` も同一条件で学習）。
3. **軸 C（P0 + P1 公平探索圧縮）**：`d_idx<=4` クランプバグ修正（6 ペア `d1..d6` 対応）＋ 終盤接近動的 MPC `t`（`--mpc-dyn-t`）＋ パリティ一致 `d/2` MPC（`--mpc-mode eg_d2`）＋ 評価マージン連動 LMR（`--mg-lmr`）。
4. **既存成果物の完全保護**：既存 `run_egaroucid_25m_v20/`・`v33/`・`cache_bin/` は読み取り専用で再利用し、全成果物を `run_egaroucid_25m_v40/` へ保存します。

## Cell 1: Google Drive マウント・環境確認・段階ゲート設定

In [ ]:
import os
import torch
from google.colab import drive

drive.mount("/content/drive")

# === 実行モード切替フラグ ===
POC_ONLY = False          # True: 300万局面クイック確認 / False: 全2,551万局面本番
TRAIN_4CH_LOC = True      # True: 比較用 4ch 30-blk local_relu=1 も同時学習
LAMBDA_BLEND = 0.65       # 再ラベルブレンド係数 (ply 12..49)

RUN_NAME = "run_egaroucid_25m_v40"
V20_RUN_DIR = "/content/drive/MyDrive/p2c64_runs/run_egaroucid_25m_v20"
V33_RUN_DIR = "/content/drive/MyDrive/p2c64_runs/run_egaroucid_25m_v33"
DRIVE_DATA_DIR = "/content/drive/MyDrive/p2c64_data"
DRIVE_CACHE_DIR = f"{DRIVE_DATA_DIR}/cache_bin"
DRIVE_WORK_DIR = f"/content/drive/MyDrive/p2c64_runs/{RUN_NAME}"
LOCAL_DATA_DIR = "/content/data"
REPO_DIR = "/content/p2c64-othello"

os.makedirs(DRIVE_WORK_DIR, exist_ok=True)
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

VRAM_GB = 0.0
if torch.cuda.is_available():
    VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU     : {torch.cuda.get_device_name(0)} ({VRAM_GB:.1f} GiB VRAM)")
    !nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
else:
    print("Running on CPU runtime (recommended for Cell 3 OpenMP relabel & Cell 6A/6B matches)")

TRAIN_BATCH = 32768 if VRAM_GB >= 20.0 else 16384
print(f"Config  : RUN_NAME={RUN_NAME} | POC_ONLY={POC_ONLY} | TRAIN_4CH_LOC={TRAIN_4CH_LOC} | lambda={LAMBDA_BLEND} | bs={TRAIN_BATCH}")
!lscpu | grep -E "Model name|CPU\(s\):|L1d|L2|L3" | head -n 10

## Cell 2: `origin/main` 取得 ＆ 純スカラービルド・単体テスト・42 カーネル 0-SIMD 監査・`SWAR_V40` レイテンシ確認

In [ ]:
%%bash
set -euo pipefail
REPO_URL="https://github.com/Taichi111009/p2c64-othello.git"
REPO_DIR="/content/p2c64-othello"

if [ -d "$REPO_DIR/.git" ]; then
    cd "$REPO_DIR" && git fetch origin && git pull --ff-only origin main
else
    git clone "$REPO_URL" "$REPO_DIR"
fi

cd "$REPO_DIR"
make clean all test audit bench
ls -lh build/

## Cell 3 (Gate 0): OpenMP 固定深さ `d6` サイドカー再ラベル（`relabel_dataset`, `--resume` 対応）

In [ ]:
import os
import shutil
import subprocess

local_tr = f"{LOCAL_DATA_DIR}/train.bin"
local_va = f"{LOCAL_DATA_DIR}/val.bin"
local_raw = f"{LOCAL_DATA_DIR}/raw_ntuple64.p2c"
local_v33 = f"{LOCAL_DATA_DIR}/waca_g2_v33.p2c"

for src, dst in [
    (f"{DRIVE_CACHE_DIR}/train.bin", local_tr),
    (f"{DRIVE_CACHE_DIR}/val.bin", local_va),
    (f"{V20_RUN_DIR}/raw_ntuple64.p2c", local_raw),
    (f"{V33_RUN_DIR}/waca_g2_v33.p2c", local_v33),
]:
    if not os.path.exists(dst):
        assert os.path.exists(src), f"Required artifact missing on Drive: {src}"
        print(f"Copying {src} -> {dst} ...")
        shutil.copy2(src, dst)
    else:
        print(f"[OK] Local SSD cache ready: {dst}")

tag = "poc3m" if POC_ONLY else "full"
rel_tr_drive = f"{DRIVE_WORK_DIR}/relabel_d6_train_{tag}.i16"
rel_va_drive = f"{DRIVE_WORK_DIR}/relabel_d6_val_{tag}.i16"

max_tr_args = ["--max-samples", "3000000"] if POC_ONLY else []
max_va_args = ["--max-samples", "200000"] if POC_ONLY else []

for label, in_bin, out_i16, extra in [
    ("VAL", local_va, rel_va_drive, max_va_args),
    ("TRAIN", local_tr, rel_tr_drive, max_tr_args),
]:
    cmd = [
        f"{REPO_DIR}/build/relabel_dataset",
        "--in", in_bin,
        "--out", out_i16,
        "--model", local_v33,
        "--depth", "6",
        "--ply-min", "12",
        "--ply-max", "49",
        "--mpc", "--mpc-t", "1.5",
        "--gate-mode", "wide_safe_bnd",
        "--m25", "2.5", "--m3", "1.5", "--tt-guard", "5.5",
        "--tt-mb", "32",
        "--threads", "0",
        "--shard-size", "1000000",
        "--resume",
    ] + extra
    print(f"\n=== [Gate 0] Relabeling {label} ({tag}) ===")
    subprocess.run(cmd, check=True)

print("\n[Gate 0 DONE] Sidecar files ready:", rel_tr_drive, rel_va_drive)

## Cell 4 (Gate 1): GPU End-to-End 共同学習（`SWAR_V40` ＋ 対照 `RAW_RELABEL`）＆ `d4→d8` $\sigma$ 自動ゲート判定

In [ ]:
import csv
import subprocess

tag = "poc3m" if POC_ONLY else "full"
rel_tr = f"{DRIVE_WORK_DIR}/relabel_d6_train_{tag}.i16"
rel_va = f"{DRIVE_WORK_DIR}/relabel_d6_val_{tag}.i16"
max_tr_args = ["--max-train", "3000000", "--max-val", "200000"] if POC_ONLY else []

p_swar_8c = f"{DRIVE_WORK_DIR}/swar_v40_8ch_30blk.p2c"
p_swar_4c = f"{DRIVE_WORK_DIR}/swar_v40_4ch_30blk_loc.p2c"
p_raw_rel = f"{DRIVE_WORK_DIR}/raw_ntuple64_relabel.p2c"

# 1) Primary: SWAR_V40 8ch 30-blk local_relu=0 (Tier-1 Unfrozen E2E)
subprocess.run([
    "python3", f"{REPO_DIR}/python/train_swar_v40.py",
    "--mode", "swar_v40",
    "--train", f"{LOCAL_DATA_DIR}/train.bin",
    "--val", f"{LOCAL_DATA_DIR}/val.bin",
    "--init-raw", f"{LOCAL_DATA_DIR}/raw_ntuple64.p2c",
    "--relabel-train", rel_tr,
    "--relabel-val", rel_va,
    "--lambda", str(LAMBDA_BLEND),
    "--emb-ch", "8", "--num-blocks", "30", "--local-relu", "0",
    "--float-epochs", "7", "--qat-epochs", "2",
    "--batch-size", str(TRAIN_BATCH),
    "--t1-lr", "2e-3", "--emb-lr", "6e-3", "--head-lr", "4e-3",
    "--out", p_swar_8c,
    "--metrics-out", f"{DRIVE_WORK_DIR}/metrics_v40_8ch_30blk",
] + max_tr_args, check=True)

# 2) Optional Comparison: SWAR_V40 4ch 30-blk local_relu=1
if TRAIN_4CH_LOC:
    subprocess.run([
        "python3", f"{REPO_DIR}/python/train_swar_v40.py",
        "--mode", "swar_v40",
        "--train", f"{LOCAL_DATA_DIR}/train.bin",
        "--val", f"{LOCAL_DATA_DIR}/val.bin",
        "--init-raw", f"{LOCAL_DATA_DIR}/raw_ntuple64.p2c",
        "--relabel-train", rel_tr,
        "--relabel-val", rel_va,
        "--lambda", str(LAMBDA_BLEND),
        "--emb-ch", "4", "--num-blocks", "30", "--local-relu", "1",
        "--float-epochs", "7", "--qat-epochs", "2",
        "--batch-size", str(TRAIN_BATCH),
        "--t1-lr", "2e-3", "--emb-lr", "6e-3", "--head-lr", "4e-3",
        "--out", p_swar_4c,
        "--metrics-out", f"{DRIVE_WORK_DIR}/metrics_v40_4ch_30blk_loc",
    ] + max_tr_args, check=True)

# 3) Control Baseline: RAW_NTUPLE_64 on identical relabeled targets
subprocess.run([
    "python3", f"{REPO_DIR}/python/train_swar_v40.py",
    "--mode", "raw_relabel",
    "--train", f"{LOCAL_DATA_DIR}/train.bin",
    "--val", f"{LOCAL_DATA_DIR}/val.bin",
    "--init-raw", f"{LOCAL_DATA_DIR}/raw_ntuple64.p2c",
    "--relabel-train", rel_tr,
    "--relabel-val", rel_va,
    "--lambda", str(LAMBDA_BLEND),
    "--float-epochs", "7", "--qat-epochs", "2",
    "--batch-size", str(TRAIN_BATCH),
    "--t1-lr", "2e-3",
    "--out", p_raw_rel,
    "--metrics-out", f"{DRIVE_WORK_DIR}/metrics_v40_raw_relabel",
] + max_tr_args, check=True)

# 4) Gate 0/1 Automatic MPC sigma Verification (d4 -> d8, ply 30-39 < 2.516 target)
cal_swar_d4 = f"{DRIVE_WORK_DIR}/mpc_cal_swar_v40_d4.csv"
cal_raw_d4 = f"{DRIVE_WORK_DIR}/mpc_cal_raw_d4.csv"
cal_swar_egd2 = f"{DRIVE_WORK_DIR}/mpc_cal_swar_v40_egd2.csv"
cal_raw_egd2 = f"{DRIVE_WORK_DIR}/mpc_cal_raw_egd2.csv"

subprocess.run([
    f"{REPO_DIR}/build/measure_mpc_sigma",
    "--data", f"{LOCAL_DATA_DIR}/val.bin", "--num-pos", "2000",
    "--models", p_swar_8c, "--calibrate-out", cal_swar_d4,
], check=True)
subprocess.run([
    f"{REPO_DIR}/build/measure_mpc_sigma",
    "--data", f"{LOCAL_DATA_DIR}/val.bin", "--num-pos", "2000",
    "--models", f"{LOCAL_DATA_DIR}/raw_ntuple64.p2c", "--calibrate-out", cal_raw_d4,
], check=True)
subprocess.run([
    f"{REPO_DIR}/build/measure_mpc_sigma",
    "--data", f"{LOCAL_DATA_DIR}/val.bin", "--num-pos", "800",
    "--models", p_swar_8c, "--calibrate-out", cal_swar_egd2, "--cal-eg-d2",
], check=True)
subprocess.run([
    f"{REPO_DIR}/build/measure_mpc_sigma",
    "--data", f"{LOCAL_DATA_DIR}/val.bin", "--num-pos", "800",
    "--models", f"{LOCAL_DATA_DIR}/raw_ntuple64.p2c", "--calibrate-out", cal_raw_egd2, "--cal-eg-d2",
], check=True)

sig_30_39 = None
with open(cal_swar_d4) as f:
    for r in csv.DictReader(f):
        if int(r["d_shallow"]) == 4 and int(r["d_deep"]) == 8 and int(r["ply_lo"]) == 30:
            sig_30_39 = float(r["sigma"])
            break

print(f"\n=== [Gate 0/1 Auto Verdict] SWAR_V40 d4->d8 ply 30..39 sigma = {sig_30_39:.4f} (RAW baseline = 2.5160) ===")
if sig_30_39 is not None and sig_30_39 < 2.516:
    print("[GATE 0/1 PASS] Midgame search consistency improved below RAW baseline! Proceed to Cell 5 & Cell 6A.")
else:
    print("[GATE 0/1 WARNING/KILL CHECK] sigma >= 2.516. Review lambda or relabel depth before Cell 6B.")

## Cell 5: 全 6 世代（v2.0 / v3.0 / v3.1 / v3.2 / v3.3 / v4.0）C++/Python ビット完全一致 ＆ 16 対称監査

In [ ]:
import subprocess

cmd = [
    "python3", f"{REPO_DIR}/python/test_bit_exact_io.py",
    "--synthetic",
    "--data", f"{LOCAL_DATA_DIR}/val.bin",
    "--num-pos", "1000",
    "--work-dir", DRIVE_WORK_DIR,
    "--models", f"{LOCAL_DATA_DIR}/raw_ntuple64.p2c", f"{LOCAL_DATA_DIR}/waca_g2_v33.p2c",
]
subprocess.run(cmd, check=True)

## Cell 6A (Gate 1 固定ノード ＋ Gate 2 20ms 公平アブレーション — CPU ランタイム推奨)

In [ ]:
import subprocess

ME = f"{REPO_DIR}/build/match_engine"
RAW = f"{LOCAL_DATA_DIR}/raw_ntuple64.p2c"
RAW_REL = f"{DRIVE_WORK_DIR}/raw_ntuple64_relabel.p2c"
SWAR = f"{DRIVE_WORK_DIR}/swar_v40_8ch_30blk.p2c"
CAL_SWAR_D4 = f"{DRIVE_WORK_DIR}/mpc_cal_swar_v40_d4.csv"
CAL_RAW_D4 = f"{DRIVE_WORK_DIR}/mpc_cal_raw_d4.csv"
CAL_SWAR_EG = f"{DRIVE_WORK_DIR}/mpc_cal_swar_v40_egd2.csv"
CAL_RAW_EG = f"{DRIVE_WORK_DIR}/mpc_cal_raw_egd2.csv"
CSV_6A = f"{DRIVE_WORK_DIR}/match_gate1_gate2_20ms.csv"

# 1) Gate 1: Fixed 20k Nodes vs RAW_NTUPLE_64 (Pass criterion: >= +40 Elo)
print("=== [Gate 1] Fixed 20,000 Nodes: SWAR_V40 vs RAW_NTUPLE_64 (400 games) ===")
subprocess.run([
    ME, "--p1", SWAR, "--p2", RAW, "--mode", "nodes", "--limit", "20000",
    "--pairs", "200", "--tt-mb", "256", "--mpc", "--mpc-t", "1.80",
    "--mo", "b4_hybrid", "--csv", CSV_6A,
], check=True)

# 2) Gate 2 (20ms b4_hybrid) #1: Legacy MPC
print("\n=== [Gate 2-A] 20ms b4_hybrid Legacy MPC: SWAR_V40 vs RAW ===")
subprocess.run([
    ME, "--p1", SWAR, "--p2", RAW, "--mode", "time", "--limit", "20",
    "--pairs", "200", "--tt-mb", "256", "--mpc", "--mpc-t", "1.80",
    "--mpc-mode", "legacy", "--mo", "b4_hybrid", "--csv", CSV_6A,
], check=True)

# 3) Gate 2 (20ms b4_hybrid) #2: Calibrated t=1.28 + mpc-dyn-t
print("\n=== [Gate 2-B] 20ms b4_hybrid Calibrated t=1.28 + dyn_t: SWAR_V40 vs RAW ===")
subprocess.run([
    ME, "--p1", SWAR, "--p2", RAW, "--mode", "time", "--limit", "20",
    "--pairs", "200", "--tt-mb", "256", "--mpc", "--mpc-t", "1.28",
    "--mpc-mode", "cal", "--mpc-dyn-t",
    "--mpc-cal-p1", CAL_SWAR_D4, "--mpc-cal-p2", CAL_RAW_D4,
    "--mo", "b4_hybrid", "--csv", CSV_6A,
], check=True)

# 4) Gate 2 (20ms b4_hybrid) #3: eg_d2 + dyn_t + D3_mgLMR_m4
print("\n=== [Gate 2-C] 20ms b4_hybrid eg_d2 + dyn_t + mgLMR(4.0): SWAR_V40 vs RAW ===")
subprocess.run([
    ME, "--p1", SWAR, "--p2", RAW, "--mode", "time", "--limit", "20",
    "--pairs", "200", "--tt-mb", "256", "--mpc", "--mpc-t", "1.28",
    "--mpc-mode", "eg_d2", "--mpc-dyn-t", "--mg-lmr", "--mg-lmr-margin", "4.0",
    "--mpc-cal-p1", CAL_SWAR_EG, "--mpc-cal-p2", CAL_RAW_EG,
    "--mo", "b4_hybrid", "--csv", CSV_6A,
], check=True)

# 5) Control Isolation: SWAR_V40 vs RAW_NTUPLE_64_RELABEL (20ms b4_hybrid)
print("\n=== [Control] 20ms b4_hybrid: SWAR_V40 vs RAW_NTUPLE_64_RELABEL ===")
subprocess.run([
    ME, "--p1", SWAR, "--p2", RAW_REL, "--mode", "time", "--limit", "20",
    "--pairs", "200", "--tt-mb", "256", "--mpc", "--mpc-t", "1.80",
    "--mpc-mode", "legacy", "--mo", "b4_hybrid", "--csv", CSV_6A,
], check=True)

## Cell 6B (Gate 2 超深読み本決戦: 50ms / 100ms, 各 600 局 — `b4_hybrid` / `static` 双方)

In [ ]:
import subprocess

ME = f"{REPO_DIR}/build/match_engine"
RAW = f"{LOCAL_DATA_DIR}/raw_ntuple64.p2c"
RAW_REL = f"{DRIVE_WORK_DIR}/raw_ntuple64_relabel.p2c"
SWAR = f"{DRIVE_WORK_DIR}/swar_v40_8ch_30blk.p2c"
CAL_SWAR_EG = f"{DRIVE_WORK_DIR}/mpc_cal_swar_v40_egd2.csv"
CAL_RAW_EG = f"{DRIVE_WORK_DIR}/mpc_cal_raw_egd2.csv"
CSV_6B = f"{DRIVE_WORK_DIR}/match_gate2_main_50ms_100ms.csv"

# Primary Evaluation: SWAR_V40 vs RAW_NTUPLE_64 (50ms & 100ms, 600 games each, b4_hybrid & static)
for tc in (50, 100):
    for mo in ("b4_hybrid", "static"):
        print(f"\n=== [MAIN {tc}ms {mo}] SWAR_V40 vs RAW_NTUPLE_64 (600 games) ===")
        subprocess.run([
            ME, "--p1", SWAR, "--p2", RAW, "--mode", "time", "--limit", str(tc),
            "--pairs", "300", "--tt-mb", "256", "--mpc", "--mpc-t", "1.28",
            "--mpc-mode", "eg_d2", "--mpc-dyn-t", "--mg-lmr", "--mg-lmr-margin", "4.0",
            "--mpc-cal-p1", CAL_SWAR_EG, "--mpc-cal-p2", CAL_RAW_EG,
            "--mo", mo, "--csv", CSV_6B,
        ], check=True)

# Secondary Evaluation: SWAR_V40 vs RAW_NTUPLE_64_RELABEL (50ms & 100ms, 600 games, b4_hybrid)
for tc in (50, 100):
    print(f"\n=== [SECONDARY {tc}ms b4_hybrid] SWAR_V40 vs RAW_NTUPLE_64_RELABEL (600 games) ===")
    subprocess.run([
        ME, "--p1", SWAR, "--p2", RAW_REL, "--mode", "time", "--limit", str(tc),
        "--pairs", "300", "--tt-mb", "256", "--mpc", "--mpc-t", "1.80",
        "--mpc-mode", "legacy", "--mo", "b4_hybrid", "--csv", CSV_6B,
    ], check=True)